# 01 — QUBO Formulation for Tokyo VRP

This notebook demonstrates the complete pipeline from a Tokyo delivery problem
to a QUBO (Quadratic Unconstrained Binary Optimization) matrix ready for
quantum solving.

## What We Cover
1. Generate a realistic Shibuya delivery dataset
2. Encode the VRP as a QUBO using Position and Route encodings
3. Visualize the QUBO matrix structure
4. Verify via brute-force that the QUBO encodes the correct optimal route
5. Compare encoding efficiency (qubits required)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from src.data.tokyo_generator import TokyoDatasetGenerator
from src.qubo.vrp_qubo import VRPQuboBuilder, VRPInstance
from src.qubo.encodings import PositionEncoding, RouteEncoding
from src.solvers.classical_baseline import brute_force_tsp

plt.style.use('dark_background')
print('Modules loaded ✅')

## 1. Generate Tokyo Delivery Data

We create a small 3-stop instance in Shibuya Ward with realistic
demands, time windows, and shortest-path distances.

In [ ]:
gen = TokyoDatasetGenerator(seed=42)
dataset = gen.generate_dataset(3, name='shibuya_3stops_demo')
instance = gen.to_vrp_instance(dataset)

print(f'Dataset: {dataset.name}')
print(f'Stops: {dataset.n_stops}')
print(f'Total demand: {dataset.total_demand} parcels')
print(f'Vehicle capacity: {dataset.vehicle_capacity}')
print()
for s in dataset.stops:
    print(f"  {s['name']:30s} | demand={s['demand']} | "
          f"time={s['time_window'][0]//60:02d}:{s['time_window'][0]%60:02d}-"
          f"{s['time_window'][1]//60:02d}:{s['time_window'][1]%60:02d} | "
          f"{s['address']}")

In [ ]:
# Distance matrix
dm = np.array(dataset.distance_matrix)
fig, ax = plt.subplots(1, 1, figsize=(6, 5))
im = ax.imshow(dm, cmap='magma')
ax.set_title('Distance Matrix (meters)', fontsize=14, fontweight='bold')
labels = ['Depot'] + [f'Stop {i+1}' for i in range(dataset.n_stops)]
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_yticklabels(labels)
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f'{dm[i,j]:.0f}', ha='center', va='center', fontsize=9,
                color='white' if dm[i,j] < dm.max()/2 else 'black')
fig.colorbar(im, ax=ax, label='Distance (m)')
plt.tight_layout()
plt.show()

## 2. QUBO Construction

We build two QUBO matrices using different encodings:
- **Position encoding**: x[stop][position] — O(n²) qubits
- **Route encoding**: x[from][to] — O(n·edges) qubits, efficient for sparse graphs

In [ ]:
# Position encoding
builder_pos = VRPQuboBuilder(instance, encoding='position')
qubo_pos = builder_pos.build()
print(f'Position Encoding:')
print(f'  Qubits: {qubo_pos.n_qubits}')
print(f'  QUBO matrix shape: {qubo_pos.Q.shape}')
print(f'  Non-zero entries: {np.count_nonzero(qubo_pos.Q)}')
print()

# Route encoding  
builder_rte = VRPQuboBuilder(instance, encoding='route')
qubo_rte = builder_rte.build()
print(f'Route Encoding:')
print(f'  Qubits: {qubo_rte.n_qubits}')
print(f'  QUBO matrix shape: {qubo_rte.Q.shape}')
print(f'  Non-zero entries: {np.count_nonzero(qubo_rte.Q)}')

In [ ]:
# Visualize QUBO matrices
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, Q, title in [(axes[0], qubo_pos.Q, f'Position Encoding ({qubo_pos.n_qubits}q)'),
                      (axes[1], qubo_rte.Q, f'Route Encoding ({qubo_rte.n_qubits}q)')]:
    Q_sym = (Q + Q.T) / 2
    im = ax.imshow(Q_sym, cmap='RdBu_r', aspect='auto')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Qubit index')
    ax.set_ylabel('Qubit index')
    fig.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('QUBO Matrix Structure', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 3. Brute-Force Verification

For this small instance, we can check every possible bitstring and confirm
the QUBO minimum corresponds to the optimal delivery route.

In [ ]:
# Brute-force optimal on position encoding
best_bits, best_energy = builder_pos.brute_force_solve()
eval_result = builder_pos.evaluate_solution(best_bits)

print(f'=== Position Encoding Brute-Force ===')
print(f'Optimal bitstring: {best_bits}')
print(f'QUBO energy: {best_energy:.2f}')
print(f'Feasible: {eval_result["feasible"]}')
print(f'Route cost: {eval_result["cost"]:.2f} meters')
print(f'Capacity used: {eval_result["demand_served"]}/{instance.capacity}')
print()

# Compare with classical brute-force TSP
classical = brute_force_tsp(instance.distance_matrix)
print(f'=== Classical Brute-Force TSP ===')
print(f'Optimal route: {classical.route}')
print(f'Route cost: {classical.cost:.2f} meters')
print()

if eval_result['feasible']:
    gap = abs(eval_result['cost'] - classical.cost) / classical.cost * 100
    print(f'✅ QUBO vs Classical gap: {gap:.2f}%')
    if gap < 1:
        print('🎯 QUBO correctly encodes the optimal route!')

## 4. Encoding Comparison

How does qubit count scale with problem size?

In [ ]:
# Compare encoding efficiency across problem sizes
sizes = [2, 3, 4, 5]
pos_qubits = []
rte_qubits = []

for n in sizes:
    ds = gen.generate_dataset(n)
    inst = gen.to_vrp_instance(ds)
    
    bp = VRPQuboBuilder(inst, encoding='position')
    pos_qubits.append(bp.build().n_qubits)
    
    br = VRPQuboBuilder(inst, encoding='route')
    rte_qubits.append(br.build().n_qubits)

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(sizes))
width = 0.35
ax.bar(x - width/2, pos_qubits, width, label='Position Encoding', color='#ff6b6b', alpha=0.9)
ax.bar(x + width/2, rte_qubits, width, label='Route Encoding', color='#4ecdc4', alpha=0.9)
ax.set_xlabel('Number of Stops', fontsize=12)
ax.set_ylabel('Qubits Required', fontsize=12)
ax.set_title('Encoding Efficiency Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(sizes)
ax.legend()
for i, (pq, rq) in enumerate(zip(pos_qubits, rte_qubits)):
    ax.text(i - width/2, pq + 0.3, str(pq), ha='center', fontsize=10)
    ax.text(i + width/2, rq + 0.3, str(rq), ha='center', fontsize=10)
plt.tight_layout()
plt.show()

print('\nSummary:')
for n, pq, rq in zip(sizes, pos_qubits, rte_qubits):
    savings = (pq - rq) / pq * 100 if pq > rq else 0
    print(f'  {n} stops: Position={pq}q, Route={rq}q ({savings:.0f}% savings)')

## 5. Ising Hamiltonian

Convert the QUBO to an Ising Hamiltonian — the native form for
quantum hardware and QAOA circuits.

In [ ]:
# Convert to Ising
J, h, offset = builder_pos.to_ising()
print(f'Ising Hamiltonian for {instance.n_stops}-stop VRP:')
print(f'  Coupling terms (J): {np.count_nonzero(J)} non-zero')
print(f'  Field terms (h): {np.count_nonzero(h)} non-zero')
print(f'  Energy offset: {offset:.2f}')
print()
print('The QAOA ansatz will optimize over this Ising cost landscape.')